# Grids

To ensure proper spatial coverage while sparing compute time, we sample one pixel per 100km² grid cell to fit the model.

For final predictions, to ensure local specificity, we extract one pixel per 1km².

Also, for final carbon accumulation predictions, we select 10 points per cell in the Bezerra et al. 2022 dataset of future land use change conditions.

These grids are used in 7_write_csv to export the final dataframe for analysis.

* **Imports:**
    * Collection 9 MapBiomas Secondary Vegetation Age
    * Collection 9 MapBiomas Land Use Land Cover
    * `categorical`
    * `distance_to_border_mask`
    * `distance_to_secondary_edge`

* **Exports:**
    * `grid_10k_amazon_secondary_edge_removed` to GEE Feature Collection
    * `grid_10k_atlantic_secondary_edge_removed` to GEE Feature Collection
    * `grid_10k_amazon_secondary` to GEE Feature Collection
    * `grid_1k_amazon_secondary` to GEE Feature Collection
    * `grid_1k_amazon_pastureland` to GEE Feature Collection
    * `grid_Bezerra_10_points` to GEE Feature Collection: Samples 10 points per 100km² grid cell from the Bezerra et al. 2022 future predictions dataset.

In [22]:
import ee
import geemap
from utils import *
initialize()

config = ProjectConfig()
roi = config.roi
data_folder = config.data_folder
last_year = config.last_year


Successfully saved authorization token.


## create_grid
Make grid to export one pixel per 10km2 and one pixel per km2.

`cell_size` = resolution in meters to sample

In [ ]:
def create_grid(image, region_name = "amazon", cell_size = 10000, file_name = None):

    biomes_fc = ee.FeatureCollection('projects/mapbiomas-workspace/AUXILIAR/biomas-2019')
    biomes = ee.Image(f"{config.data_folder}/categorical").select("biome")
    
    if region_name == "amazon":
        biome_index = 1
        biome_geom = (biomes_fc
                    .filter(ee.Filter.eq('Bioma', 'Amazônia'))
                    .geometry())

    elif region_name == "atlantic":
        biome_index = 4
        biome_geom = (biomes_fc
                    .filter(ee.Filter.eq('Bioma', 'Mata Atlântica'))
                    .geometry())

    biomes = biomes.eq(biome_index).selfMask()

    pixels_to_sample = biomes.reduceResolution(
        ee.Reducer.first(), maxPixels=65536
        ).reproject(
        crs = image.projection().getInfo()['crs'],
        crsTransform = image.projection().getInfo()['transform']
        ).updateMask(image)
    
    image_scale = round(pixels_to_sample.projection().nominalScale().getInfo())
    closest_multiple = round(cell_size / image_scale) * image_scale

    # First, sample locations based only on the age band
    grid = geemap.create_grid(pixels_to_sample.geometry(), closest_multiple, pixels_to_sample.projection())

    grid = grid.filterBounds(biome_geom.bounds())
    
    # Function to sample one point per valid cell
    def sample_cell(cell):
        sampled_fc = pixels_to_sample.stratifiedSample(
            numPoints = 1,
            classBand = 'biome',
            region = cell.geometry(),
            scale = pixels_to_sample.projection().nominalScale(),
            geometries = True,
            dropNulls = True,
            tileScale = 3
        )

        # Only return a feature if we found one
        return ee.Feature(ee.Algorithms.If(
            sampled_fc.size().gt(0),
            sampled_fc.first(),
            # Return a placeholder that we can filter out later
            ee.Feature(ee.Geometry.Point([0, 0])).set('is_null', True)
        ))

    samples = grid.map(sample_cell)

    # Filter out placeholder features before exporting
    samples = samples.filter(ee.Filter.notEquals('is_null', True))

    if file_name is None:
        return samples
    else:
        export_name = f"grid_{cell_size//1000}k_{region_name}_{file_name}"

        export_task = ee.batch.Export.table.toAsset(
            collection = samples,
            description = export_name,
            assetId = f"{config.data_folder}/{export_name}"
        )
        export_task.start()

In [ ]:
distance_to_border_mask = ee.Image(f"{data_folder}/distance_to_border_mask")
age = ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_secondary_vegetation_age_v1") \
        .select("secondary_vegetation_age_2020")\
        .updateMask(distance_to_border_mask).rename("age")

edge = ee.Image(f"{data_folder}/distance_to_secondary_edge").gt(30) # non-edge pixel mask (only those surrounded by secondary forests on all sides)
age_edge_removed = age.updateMask(edge)

pastureland = (ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_integration_v1")
            .select([f"classification_{year}" for year in config.range_1985_2020])
            .byte()
            .rename([str(year) for year in config.range_1985_2020]))
pastureland = pastureland.select("2020").eq(15).unmask(0).rename("pastureland")

In [ ]:
# create_grid(age_edge_removed, region_name = "amazon", cell_size = 10000, file_name = "secondary_edge_removed")
# create_grid(age_edge_removed, region_name = "atlantic", cell_size = 10000, file_name = "secondary_edge_removed")

# create_grid(age, region_name = "amazon", cell_size = 10000, file_name = "secondary")
# create_grid(age, region_name = "amazon", cell_size = 1000, file_name = "secondary")
# create_grid(pastureland, region_name = "amazon", cell_size = 1000, file_name = "pastureland")

## Grid for future projections - Bezerra map

Select 10 points per pixel of the Bezerra et al. 2022 projection map (10km x 10km) to make future predictions. We know how much forest area increase happened per pixel, but since we don't know exactly where within each 10km that did occur, we sample 10 random pixels to have a good representation of the diversity of local regrowth conditions in each pixel. This leads to an assumption that the area of regrowth is distributed randomly.

In [ ]:


biomes = ee.Image(f"{config.data_folder}/categorical").select("biome")
biomes = biomes.eq(1).selfMask()

scenarios = ["SSP1_RCP19", "SSP2_RCP45", "SSP3_RCP70"]

scenarios_growth = ee.Image(0)
for scenario in scenarios:
    scenario_2015 = ee.Image(f"projects/forestregrowth/assets/projections/forest_2015_{scenario}")
    scenario_2050 = ee.Image(f"projects/forestregrowth/assets/projections/forest_2050_{scenario}")

    # get the area increase per 10km grid cell
    delta = scenario_2050.subtract(scenario_2015)

    scenario_growth = delta.updateMask(delta.gt(0).And(biomes)).rename(f'growth_{scenario}').unmask(0)

    scenarios_growth = scenarios_growth.add(scenario_growth)

# all pixels that show any increase in forest cover from 2015 to 2050 in any scenario (will be used for future predictions)
growth_mask = scenarios_growth.gt(0).rename("mask")

In [42]:
biome_geom = (ee.FeatureCollection('projects/mapbiomas-workspace/AUXILIAR/biomas-2019')
            .filter(ee.Filter.eq('Bioma', 'Amazônia'))
            .geometry())

pixels_to_sample = growth_mask.reduceResolution(
    ee.Reducer.first(), maxPixels = 65536
    ).reproject(
    crs = age.projection().getInfo()['crs'],
    crsTransform = age.projection().getInfo()['transform']
    )

grid = geemap.create_grid(scenario_2015.geometry(), scenario_2015.projection().nominalScale(), scenario_2015.projection())
grid = grid.filterBounds(biome_geom)

# Function to sample one point per valid cell
def sample_cell(cell):
    sampled_fc = pixels_to_sample.stratifiedSample(
        numPoints = 10,
        classBand = 'mask',
        region = cell.geometry(),
        scale = pixels_to_sample.projection().nominalScale(),
        geometries = True,
        dropNulls = True,
        tileScale = 3
    )

    # Only return a feature if we found one
    return ee.Feature(ee.Algorithms.If(
        sampled_fc.size().gt(0),
        sampled_fc.first(),
        # Return a placeholder that we can filter out later
        ee.Feature(ee.Geometry.Point([0, 0])).set('is_null', True)
    ))

samples = grid.map(sample_cell)

# Filter out placeholder features before exporting
samples = samples.filter(ee.Filter.notEquals('is_null', True))

export_task = ee.batch.Export.table.toAsset(
    collection = samples,
    description = "grid_Bezerra_10_points",
    assetId = f"{config.data_folder}/grid_Bezerra_10_points"
)
export_task.start()